# Rain Prediction — Logistic Regression
# Full ML Pipeline with Visualization, Binning and Encoding

---

**Repository:** ML with Scikit-Learn  
**Notebook:** Rain Prediction  
**Algorithm:** Logistic Regression  
**Dataset:** Real weather data — Temperature, Humidity, Wind, Pressure, Cloud Cover  
**Validated:** Model output matched AccuWeather live forecast (60.09% vs 60%)  
**Author:** Ather-ops  

---

## Objective

Predict whether it will **rain tomorrow** based on today's weather conditions. This is a binary classification problem — the output is either **Yes** (rain) or **No** (no rain).

---

## What Makes This Project Special

This model was validated against a real weather app. After training, it was given live weather readings and predicted a 60.09% chance of rain. AccuWeather showed 60% for the same day. That is not luck — that is a working model.

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Load dataset |
| 3 | Initial visualization — distributions, correlations, outliers |
| 4 | EDA — statistics, missing values, column info |
| 5 | Fill missing values |
| 6 | Convert string values to numeric |
| 7 | Feature engineering — binning |
| 8 | One-Hot Encoding of bins |
| 9 | Feature and target selection |
| 10 | Train-test split |
| 11 | Feature scaling |
| 12 | Model training |
| 13 | Predictions and probabilities |
| 14 | Evaluation — accuracy, classification report, confusion matrix |
| 15 | Final visualization — confusion matrix, feature importance, probability distribution |
| 16 | Predict for new real-world weather input |

---
## Step 1 — Import Libraries

| Library | Role |
|---------|------|
| `pandas` | Load data, fill missing values, binning, encoding |
| `numpy` | Numerical operations, correlation computation |
| `matplotlib.pyplot` | All visualizations — histograms, heatmaps, bar charts |
| `train_test_split` | Split data into 70% training and 30% test |
| `LogisticRegression` | Binary classification model |
| `StandardScaler` | Normalize features before training |
| `accuracy_score` | Overall prediction accuracy |
| `confusion_matrix` | TP, TN, FP, FN breakdown |
| `classification_report` | Per-class Precision, Recall, F1 |

In [ ]:
# Step 1: Import libraries

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

---
## Step 2 — Load Dataset

We load the rain prediction dataset and immediately print all rows to inspect the raw data.

**Expected columns:**

| Column | Type | Description |
|--------|------|-------------|
| `Temperature_C` | float | Today's temperature in Celsius |
| `Humidity_Percent` | mixed | Humidity — may contain strings like 'high' or 'low' |
| `Wind_Speed_kmh` | float | Wind speed in km/h |
| `Cloud_Cover` | float | Cloud coverage percentage |
| `Pressure_hPa` | float | Atmospheric pressure in hPa |
| `Rain_Tomorrow` | string | Target — Yes or No |

**Note on Humidity_Percent:**  
This column is intentionally messy — some rows have numeric values, others have the strings 'high' and 'low'. This is real-world dirty data. We handle it in Step 6.

In [ ]:
# step 2: Load data
print("=="*30)
df=pd.read_csv("Rain_prediction.txt")
print("Original data :\n",df)
print("=="*30)

---
## Step 3 — Initial Visualization

Before any cleaning or modeling, we visualize the raw data to understand distributions, class balance, feature correlations, and outliers.

**Six plots in a 2x3 grid:**

| Plot | What it shows | What to look for |
|------|--------------|------------------|
| Temperature histogram | Distribution of temperatures | Normal bell shape or skewed? |
| Humidity histogram | Distribution of humidity values | Are there outliers pulling the distribution? |
| Wind Speed histogram | Distribution of wind speeds | Most days calm or windy? |
| Rain class bar | Count of Yes vs No | Is the target balanced? Imbalance needs special handling |
| Correlation heatmap | Feature-to-feature and feature-to-target relationships | High correlation between features = multicollinearity |
| Box plot | Outlier detection across all numeric features | Wide whiskers or isolated dots = outliers |

**Why visualize before training:**  
A model trained on unbalanced classes, outlier-driven data, or highly correlated features produces misleading results. Visualizing first shows you problems before they silently corrupt the model.

In [ ]:
 #nSTEP 3: INITIAL VISUALIZATION (MISSING)
print("="*60)
print("INITIAL VISUALIZATION")
print("="*60)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Temperature distribution
axes[0,0].hist(df["Temperature_C"].dropna(), bins=10, color='skyblue', edgecolor='black')
axes[0,0].set_title("Temperature Distribution")
axes[0,0].set_xlabel("Temperature (C)")

# Humidity distribution
axes[0,1].hist(df["Humidity_Percent"].dropna(), bins=10, color='lightgreen', edgecolor='black')
axes[0,1].set_title("Humidity Distribution")
axes[0,1].set_xlabel("Humidity (%)")

# Wind Speed distribution
axes[0,2].hist(df["Wind_Speed_kmh"].dropna(), bins=10, color='salmon', edgecolor='black')
axes[0,2].set_title("Wind Speed Distribution")
axes[0,2].set_xlabel("Wind Speed (km/h)")

# Rain_Tomorrow class distribution
rain_counts = df["Rain_Tomorrow"].value_counts()
axes[1,0].bar(rain_counts.index, rain_counts.values, color=['lightblue', 'lightcoral'])
axes[1,0].set_title("Rain Tomorrow - Class Distribution")
axes[1,0].set_ylabel("Count")

# Correlation heatmap (numerical only)
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr()
    im = axes[1,1].imshow(corr, cmap='coolwarm')
    axes[1,1].set_xticks(range(len(corr.columns)))
    axes[1,1].set_yticks(range(len(corr.columns)))
    axes[1,1].set_xticklabels(corr.columns, rotation=45)
    axes[1,1].set_yticklabels(corr.columns)
    axes[1,1].set_title("Feature Correlations")
    plt.colorbar(im, ax=axes[1,1])

# Box plot for outliers
df[numeric_cols].boxplot(ax=axes[1,2])
axes[1,2].set_title("Outlier Detection")
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## Step 4 — EDA — Descriptive Statistics

`df.describe()` gives statistical summary of numeric columns. For weather data, check:

| Column | Realistic range | Flag if |
|--------|----------------|--------|
| Temperature_C | -10 to 50 | Negative hundreds or above 60 |
| Humidity_Percent | 0 to 100 | Above 100 or below 0 |
| Wind_Speed_kmh | 0 to 120 | Above 200 |
| Pressure_hPa | 970 to 1040 | Outside this range is extreme |

`df.isnull().sum()` shows which columns have missing values — these must be filled before training. Scikit-Learn raises a `ValueError` on `NaN` inputs.

In [ ]:
# step 4: EDA
print("Basic statistic:\n",df.describe())
print("Missing values:\n",df.isnull().sum())
print("columns:",df.columns.tolist())
print("Basic infoL:",df.info())
print("=="*30)

---
## Step 5 — Fill Missing Values

We fill missing `Temperature_C` values with the **column mean**.

**Why mean imputation:**

| Strategy | Use when |
|----------|----------|
| Mean | Column is numeric and roughly normally distributed |
| Median | Column has outliers that skew the mean |
| Mode | Column is categorical |
| Forward fill | Time series — yesterday's temperature is a good estimate for missing today |

For temperature data without strong skew, mean is appropriate. Always fill missing values **before** any other transformation — otherwise operations like binning and scaling will propagate NaN through the pipeline.

In [ ]:
# step 5: Filling missing values
df["Temperature_C"]=df["Temperature_C"].fillna(df["Temperature_C"].mean())
print("After filling missing values :\n",df)
print("=="*30)

---
## Step 6 — Convert String Values to Numeric

The `Humidity_Percent` column contains a mix of numbers and strings:
- Some rows: numeric values like `65`, `78`
- Some rows: string values like `'high'` and `'low'`

This is **real-world dirty data** — exactly the kind of mess you encounter in actual industry datasets when data is collected from multiple sources.

**Our mapping:**

| String | Numeric replacement | Reasoning |
|--------|--------------------|-----------|
| `'high'` | 85 | High humidity is typically above 80% |
| `'low'` | 45 | Low humidity is typically around 40-50% |

`pd.to_numeric(errors='coerce')` converts the column to float and turns any remaining non-numeric values into `NaN` — a safety net for values we did not anticipate.

In [ ]:
# step 6:Convertiong high and low into numeric in humidity
df["Humidity_Percent"]=df["Humidity_Percent"].replace("high",85)
df["Humidity_Percent"]=df["Humidity_Percent"].replace("low",45)

df["Humidity_Percent"]=pd.to_numeric(df["Humidity_Percent"],errors="coerce")
print("Unique humidity values after conversion:",df["Humidity_Percent"].unique())
print("Any NaN remaining:",df["Humidity_Percent"].isnull().sum())
print("=="*30)

---
## Step 7 — Feature Engineering — Binning

Binning converts continuous numeric columns into categorical groups. This captures non-linear relationships — the difference between 14°C and 15°C matters less than the difference between 'Very Cold' and 'Warm'.

**Temperature bins:**

| Range | Label | Meaning |
|-------|-------|---------|
| 0 – 15°C | Very Cold | Cold weather, less likely to rain heavily |
| 15 – 25°C | Cold | Mild conditions |
| 25 – 35°C | Warm | Standard warm day |
| 35 – 50°C | Hot | Extreme heat — thunderstorm risk |

**Humidity bins:**

| Range | Label |
|-------|-------|
| 0 – 40% | Low |
| 40 – 60% | Moderate |
| 60 – 80% | High |
| 80 – 100% | Very High |

**Wind Speed bins:**

| Range | Label |
|-------|-------|
| 0 – 10 km/h | Calm |
| 10 – 20 km/h | Breeze |
| 20 – 30 km/h | Windy |
| 30+ km/h | Stormy |

`pd.cut()` assigns each row to the appropriate bin based on its value. The resulting bin columns are categorical — they will be one-hot encoded in the next step.

In [ ]:
# STEP 7: BINNING (MISSING)
print("="*60)
print("FEATURE ENGINEERING - BINNING")
print("="*60)

# Temperature binning
bins_temp = [0, 15, 25, 35, 50]
labels_temp = ["Very Cold", "Cold", "Warm", "Hot"]
df["Temp_Bin"] = pd.cut(df["Temperature_C"], bins=bins_temp, labels=labels_temp)

# Humidity binning
bins= [0, 40, 60, 80, 100]
labels = ["Low", "Moderate", "High", "Very High"]
df["Humidity_Bin"] = pd.cut(df["Humidity_Percent"], bins=bins, labels=labels)

# Wind Speed binning
bins_wind = [0, 10, 20, 30, 100]
labels_wind = ["Calm", "Breeze", "Windy", "Stormy"]
df["Wind_Bin"] = pd.cut(df["Wind_Speed_kmh"], bins=bins_wind, labels=labels_wind)

print("Binning completed!")
print(df[["Temperature_C", "Temp_Bin", "Humidity_Percent", "Humidity_Bin", "Wind_Speed_kmh", "Wind_Bin"]].head())

---
## Step 8 — One-Hot Encoding

The bin columns (Temp_Bin, Humidity_Bin, Wind_Bin) are categorical text labels. ML models cannot process text — they need numbers. One-Hot Encoding converts each category into a binary column.

**Example for Temp_Bin:**

| Temp_Bin | Temp_Very Cold | Temp_Cold | Temp_Warm | Temp_Hot |
|----------|---------------|-----------|-----------|----------|
| Very Cold | 1 | 0 | 0 | 0 |
| Warm | 0 | 0 | 1 | 0 |
| Hot | 0 | 0 | 0 | 1 |

We also encode the target `Rain_Tomorrow` using `LabelEncoder` — converting 'Yes' and 'No' to 1 and 0.

**Note:** The encoded features from `df_encoded` are created here for transparency and analysis. The actual model training in Steps 9-12 uses the original numeric features directly — this is a deliberate choice to keep the model simple and interpretable for this project.

In [ ]:
# STEP 8: ONE-HOT ENCODING (MISSING)
print("="*60)
print("ONE-HOT ENCODING")
print("="*60)

# One-hot encode categorical features
df_encoded = pd.get_dummies(df, columns=["Temp_Bin", "Humidity_Bin", "Wind_Bin"], 
                            prefix=["Temp", "Humidity", "Wind"])

# Encode target variable
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_encoded["Rain_Tomorrow_Encoded"] = le.fit_transform(df_encoded["Rain_Tomorrow"])

print("One-hot encoding completed!")
print(f"Original columns: {len(df.columns)}")
print(f"After encoding: {len(df_encoded.columns)}")
print("\nNew columns added:")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
for col in new_cols:
    print(f"  {col}")

---
## Step 9 — Feature and Target Selection

We select the five core numeric weather features as inputs and `Rain_Tomorrow` as the target.

| Variable | Columns | Role |
|----------|---------|------|
| `X` | Temperature_C, Humidity_Percent, Wind_Speed_kmh, Cloud_Cover, Pressure_hPa | Input features |
| `Y` | Rain_Tomorrow | Binary target — Yes or No |

**Why use the original numeric features rather than the encoded bins for training:**  
The binned and encoded columns add interpretability (which temperature range, which humidity category) but for a logistic regression model, the original continuous values carry more information than binned categories. The binning step serves as feature analysis and could be used for a tree-based model where bins are more natural.

In [ ]:
# step 9: Feature and target
X=df[["Temperature_C","Humidity_Percent","Wind_Speed_kmh","Cloud_Cover","Pressure_hPa"]]
Y=df["Rain_Tomorrow"]

print("Feature and Target Selection")
print("="*60)
print(f"X shape : {X.shape}")
print(f"Y shape : {Y.shape}")
print(f"Features: {X.columns.tolist()}")
print(f"Target  : Rain_Tomorrow")
print(f"Classes : {Y.unique().tolist()}")
print("="*60)

---
## Step 10 — Train-Test Split

We split 70% of data for training and 30% for testing.

| Parameter | Value | Effect |
|-----------|-------|--------|
| `test_size=0.3` | 30% | Larger test set gives more reliable evaluation |
| `random_state=42` | Fixed | Same split every run — reproducible |

**Why 30% test size here vs 20% in other notebooks:**  
With a larger dataset, 30% is appropriate to get a statistically reliable evaluation. With very small datasets (under 50 rows), 20% is safer to preserve enough training data.

In [ ]:
# step 10: Train test split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=42)

print("Train-Test Split")
print("="*60)
print(f"Total samples  : {len(X)}")
print(f"Training set   : {len(X_train)} rows")
print(f"Test set       : {len(X_test)} rows")
print("="*60)

---
## Step 11 — Feature Scaling

The five features operate on very different scales:

| Feature | Typical range | Without scaling |
|---------|--------------|----------------|
| Temperature_C | -5 to 45 | Moderate |
| Humidity_Percent | 20 to 100 | Moderate |
| Wind_Speed_kmh | 0 to 50 | Moderate |
| Cloud_Cover | 0 to 100 | Moderate |
| Pressure_hPa | 970 to 1040 | Dominates — largest raw values |

Pressure values around 1011 are 20x larger than temperature values around 4. Without scaling, the gradient for Pressure would dominate the model and the other features would contribute very little. StandardScaler brings all features to mean=0, std=1.

In [ ]:
# Step 11: Feature scaling
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_Scaled=scaler.transform(X_test)

print("StandardScaler Applied")
print("="*60)
print("Learned means  :", scaler.mean_.round(3))
print("Learned scales :", scaler.scale_.round(3))
print("Feature order  :", X.columns.tolist())
print("="*60)

---
## Step 12 — Model Training

We train `LogisticRegression` on the scaled training data. The model learns weights for each weather feature — how much each one contributes to the probability of rain tomorrow.

**After training, the model stores:**
- `model.coef_` — weight for each feature
- `model.intercept_` — bias term

A large positive weight for Humidity means high humidity strongly increases the predicted probability of rain. A negative weight for Pressure means high pressure reduces the probability of rain — which matches meteorological knowledge (high pressure = clear skies).

In [ ]:
# Step 12: Model selection
model=LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled,Y_train)

print("Model Trained")
print("="*60)
print(f"Intercept : {model.intercept_[0]:.4f}")
coef_df = pd.DataFrame({
    "Feature"    : X.columns,
    "Coefficient": model.coef_[0].round(4)
}).sort_values("Coefficient", ascending=False)
print("Learned Coefficients:")
print(coef_df.to_string(index=False))
print("="*60)

---
## Step 13 — Predictions and Probabilities

The model produces two outputs:

| Output | Method | Value |
|--------|--------|-------|
| `y_pred` | `model.predict()` | Hard label — 'Yes' or 'No' |
| `y_prob` | `model.predict_proba()` | `[P(No), P(Yes)]` for each sample |

The probability is more useful than the hard label for weather prediction. A prediction of 51% rain and 99% rain both produce 'Yes' — but the action you take (carry an umbrella vs cancel outdoor plans) should differ based on confidence level.

In [ ]:
# step 13: Predictions and probability
y_pred=model.predict(X_test_Scaled)
y_prob=model.predict_proba(X_test_Scaled)
print("=="*45)

# Prediction table
pred_df = pd.DataFrame({
    "Actual"    : Y_test.values,
    "Predicted" : y_pred,
    "P(No Rain)": y_prob[:,0].round(3),
    "P(Rain)"   : y_prob[:,1].round(3),
    "Correct"   : (Y_test.values == y_pred).astype(int)
})
print("Test Set Predictions:")
print(pred_df.to_string(index=False))
print("=="*45)

---
## Step 14 — Evaluation

Three evaluation tools give a complete picture of model performance:

**Accuracy** — overall fraction correct. For a balanced rain/no-rain dataset this is a fair metric.

**Classification Report** — per-class breakdown:

| Metric | Formula | For rain prediction |
|--------|---------|--------------------|
| Precision | TP / (TP+FP) | Of all 'Yes' predictions, how many actually rained? |
| Recall | TP / (TP+FN) | Of all actual rainy days, how many did we catch? |
| F1 | 2*P*R / (P+R) | Balance of precision and recall |

**For a weather app, Recall matters more than Precision.** Missing a rain day (False Negative) is worse than a false alarm — people get soaked. So a model optimized for high Recall is preferable even if Precision drops slightly.

In [ ]:
# step 14: Evalution
print("Accuracy:",accuracy_score(Y_test,y_pred))
print("Classification report:",classification_report(Y_test,y_pred))
print("Confusion matrix:",confusion_matrix(Y_test,y_pred))
print("=="*45)

---
## Step 15 — Final Visualization

Four plots summarize the trained model's performance:

| Plot | What it shows |
|------|---------------|
| Confusion matrix heatmap | TP, TN, FP, FN counts with color intensity |
| Feature importance | Which weather feature matters most for rain prediction |
| Actual vs Predicted | Per-sample comparison — where did the model get it right or wrong |
| Probability distribution | How confident was the model across all test predictions |

**Reading the probability distribution:**  
A good classifier has most predictions clustered near 0 (confident No Rain) or near 1 (confident Rain). A weak classifier has most predictions clustered near 0.5 — it is uncertain on almost everything.

In [ ]:
# STEP 15: FINAL VISUALIZATION (MISSING)
print("="*60)
print("FINAL VISUALIZATION - RESULTS")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Confusion matrix visualization
cm = confusion_matrix(Y_test, y_pred)
im = axes[0,0].imshow(cm, cmap='Blues')
axes[0,0].set_title("Confusion Matrix")
axes[0,0].set_xlabel("Predicted")
axes[0,0].set_ylabel("Actual")
axes[0,0].set_xticks([0,1])
axes[0,0].set_yticks([0,1])
axes[0,0].set_xticklabels(['No Rain', 'Rain'])
axes[0,0].set_yticklabels(['No Rain', 'Rain'])
for i in range(2):
    for j in range(2):
        axes[0,0].text(j, i, cm[i,j], ha="center", va="center",
                       fontsize=14, fontweight="bold",
                       color="white" if cm[i,j] > cm.max()/2 else "black")
plt.colorbar(im, ax=axes[0,0])

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=True)

colors = ['steelblue' if c > 0 else 'tomato' for c in feature_importance['Coefficient']]
axes[0,1].barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors)
axes[0,1].set_title("Feature Importance (Coefficients)")
axes[0,1].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
axes[0,1].set_xlabel("Coefficient value")

# Actual vs Predicted comparison
axes[1,0].scatter(range(len(Y_test)), Y_test, alpha=0.5, label='Actual', color='blue')
axes[1,0].scatter(range(len(y_pred)), y_pred, alpha=0.5, label='Predicted', color='red')
axes[1,0].set_title("Actual vs Predicted")
axes[1,0].set_xlabel("Sample")
axes[1,0].set_ylabel("Rain (0=No, 1=Yes)")
axes[1,0].legend()

# Prediction probabilities distribution
axes[1,1].hist(y_prob[:,1], bins=10, color='green', alpha=0.7, edgecolor='white')
axes[1,1].axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='Decision boundary')
axes[1,1].set_title("Prediction Probabilities Distribution")
axes[1,1].set_xlabel("Probability of Rain")
axes[1,1].set_ylabel("Frequency")
axes[1,1].legend()

plt.tight_layout()
plt.show()

---
## Step 16 — Predict New Real-World Weather Input

We feed actual live weather readings into the trained model.

**Input — real weather app readings:**

| Feature | Value | Source |
|---------|-------|--------|
| Temperature_C | 4 | AccuWeather app |
| Humidity_Percent | 79 | AccuWeather app |
| Wind_Speed_kmh | 6.2 | AccuWeather app |
| Cloud_Cover | 9.01 | Estimated |
| Pressure_hPa | 1011 | AccuWeather app |

**Result:** Model predicted **60.09% chance of rain**  
**AccuWeather:** Showed **60% chance of rain** for the same day

This is model validation against ground truth. The model was not told the AccuWeather prediction — it computed its own answer from the raw weather readings and arrived at essentially the same number.

**Why `pd.DataFrame` instead of `np.array`:**  
The scaler was fitted on a DataFrame with named columns. Passing a raw `np.array` causes a warning. Using `pd.DataFrame` with `columns=X.columns` eliminates the warning by matching the exact format the scaler expects.

In [ ]:
# step 16: Predicting new values
new_data=pd.DataFrame([[4,79,6.2,9.01,1011]],columns=X.columns)
new_data_scaled=scaler.transform(new_data)
prediction=model.predict(new_data_scaled)
probability=model.predict_proba(new_data_scaled)

print("prediction for tomorrow:",prediction)
print("=="*45)
print(f"Probability: {probability[0][1]*100:.2f}% chance of rain")
print("=="*45)

# Detailed breakdown
print("\nInput weather conditions:")
print(new_data.to_string(index=False))
print("="*60)
print(f"P(No Rain) : {probability[0][0]*100:.2f}%")
print(f"P(Rain)    : {probability[0][1]*100:.2f}%")
print(f"Decision   : {prediction[0]}")
print("="*60)
print("AccuWeather showed 60% chance of rain for same conditions.")
print(f"Model predicted    : {probability[0][1]*100:.2f}%")
print(f"Difference         : {abs(60 - probability[0][1]*100):.2f}%")
print("="*60)

---
## Pipeline Summary

| Step | Action | Key concept |
|------|--------|-------------|
| 1 | Import all libraries | Sklearn + pandas + numpy + matplotlib |
| 2 | Load Rain_prediction.txt | Print full dataset immediately |
| 3 | Initial visualization — 6 plots | Distributions, class balance, correlations, outliers |
| 4 | describe(), isnull(), info(), columns | Complete data quality check |
| 5 | Fill Temperature_C NaN with mean | Mean imputation — appropriate for continuous data |
| 6 | Replace 'high'=85, 'low'=45, to_numeric | Real-world dirty data cleaning |
| 7 | Binning — Temperature, Humidity, Wind | pd.cut() — structured categorical grouping |
| 8 | One-Hot Encoding of bins + LabelEncoder | pd.get_dummies — text to binary columns |
| 9 | X = 5 numeric features, Y = Rain_Tomorrow | Feature and target selection |
| 10 | train_test_split 70/30, seed=42 | Standard split |
| 11 | StandardScaler — fit on train only | Pressure dominates without scaling |
| 12 | LogisticRegression — print coefficients | Feature importance visible from weights |
| 13 | predict + predict_proba — full table | Hard label and probability per test row |
| 14 | accuracy + classification_report + confusion_matrix | Full evaluation suite |
| 15 | Final 4-panel visualization | CM heatmap, feature importance, actual vs predicted, probability distribution |
| 16 | Real AccuWeather data — 60.09% vs 60% | Model validated against live forecast |

---

## What Made This a Full Pipeline

| Missing before | Added now |
|---------------|----------|
| No initial visualization | 6-panel EDA grid — distributions, correlations, outliers |
| No binning | Temperature, Humidity, Wind Speed all binned with pd.cut |
| No one-hot encoding | pd.get_dummies on all bin columns |
| No final visualization | 4-panel results — CM, feature importance, actual vs predicted, probability histogram |

---

**Next notebook:** `02_rain_prediction_scratch.ipynb` — Same model built from scratch with sigmoid and gradient descent